In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Dataset Generator for Composite DNA - Eta-Based Variable Ratio 2-Mix
# ## Extended alphabet with variable mixture ratios (η=0.2, ℓ∈{-2,-1,0,1,2})

# =============================================================================
# CELL 1: IMPORTS
# =============================================================================
import random
import numpy as np
import pickle
import json
import os
from collections import Counter
from datetime import datetime
import time


# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
ERROR_MODEL = "organick"  # Options: "erlich", "grass", "organick"

# ------------------- ETA-BASED ALPHABET PARAMETERS -------------------
ETA = 0.2  # Step size for mixture ratios
ELL_VALUES = [-2, -1, 0, 1, 2]  # Offset values: gives ratios 0.1/0.9, 0.3/0.7, 0.5/0.5, 0.7/0.3, 0.9/0.1

# Calculate vocab size: 4 pure + 6 pairs × len(ELL_VALUES)
NUM_PURE_BASES = 4
NUM_TWO_MIX_PAIRS = 6
VOCAB_SIZE = NUM_PURE_BASES + NUM_TWO_MIX_PAIRS * len(ELL_VALUES)  # 4 + 30 = 34

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {"full_length": 152, "index_length": 16, "seq_length": 136, "name": "EZ17"},
    "grass": {"full_length": 117, "index_length": 13, "seq_length": 104, "name": "G15"},
    "organick": {"full_length": 110, "index_length": 33, "seq_length": 77, "name": "O17"}
}

CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_specs": ERROR_MODEL_SPECS[ERROR_MODEL],
    
    # Eta-Based Alphabet Parameters
    "eta": ETA,
    "ell_values": ELL_VALUES,
    "alphabet_mode": f"eta{ETA}_ell{len(ELL_VALUES)}",
    
    # Dataset Parameters
    "num_samples": 100000,
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    "max_coverage": 50,
    
    # Vocabulary
    "vocab_size": VOCAB_SIZE,
    
    # Output Directory
    "dataset_dir": "./dataset",
    
    # Reproducibility
    "seed": 42
}

# Create dataset name
dataset_name = f"dna_{CONFIG['error_specs']['name']}_eta{ETA}"
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{dataset_name}_"
                          f"{CONFIG['num_samples']}_{CONFIG['max_coverage']}.pkl")

os.makedirs(CONFIG['dataset_dir'], exist_ok=True)

print(f"{'='*60}")
print(f"📋 DATASET GENERATION CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Model: {CONFIG['error_model'].upper()} ({CONFIG['error_specs']['name']})")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Eta: {ETA}, Ell Values: {ELL_VALUES}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Theoretical Capacity: {np.log2(CONFIG['vocab_size']):.4f} bits/position")
print(f"   Num Samples: {CONFIG['num_samples']:,}")
print(f"   Max Coverage: {CONFIG['max_coverage']}")
print(f"   Output Path: {CONFIG['dataset_path']}")
print(f"{'='*60}")


# =============================================================================
# CELL 3: SEED FOR REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")


# =============================================================================
# CELL 4: ETA-BASED COMPOSITE DNA ALPHABET DEFINITIONS
# =============================================================================

def build_eta_based_alphabet(eta, ell_values):
    """
    Build the eta-based composite alphabet with variable mixture ratios.
    
    For each two-nucleotide pair (b1, b2), creates symbols with ratios:
        (0.5 + ℓ*η, 0.5 - ℓ*η) for each ℓ in ell_values
    
    Args:
        eta: Step size for mixture ratios (e.g., 0.2)
        ell_values: List of offset values (e.g., [-2, -1, 0, 1, 2])
    
    Returns:
        composite_map: Dict mapping symbol names to (nucleotide, probability) pairs
        symbol_to_idx: Dict mapping symbol names to class indices
        ideal_vectors: List of [A_prob, C_prob, G_prob, T_prob] vectors
    """
    
    # Pure bases: indices 0-3
    composite_map = {
        'A': [('A', 1.0)],
        'C': [('C', 1.0)],
        'G': [('G', 1.0)],
        'T': [('T', 1.0)],
    }
    
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
    ]
    
    # Define the 6 two-nucleotide pairs and their vector positions
    # Each entry: (pair_name, base1, base2, vec_idx1, vec_idx2)
    two_mix_pairs = [
        ('B1', 'A', 'C', 0, 1),  # A|C
        ('B2', 'A', 'G', 0, 2),  # A|G
        ('B3', 'A', 'T', 0, 3),  # A|T
        ('B4', 'C', 'G', 1, 2),  # C|G
        ('B5', 'C', 'T', 1, 3),  # C|T
        ('B6', 'G', 'T', 2, 3),  # G|T
    ]
    
    current_idx = 4  # Start after pure bases
    
    for pair_name, base1, base2, idx1, idx2 in two_mix_pairs:
        for ell in ell_values:
            # Calculate mixture ratios
            prob1 = 0.5 + ell * eta  # Probability for base1
            prob2 = 0.5 - ell * eta  # Probability for base2
            
            # Clamp to valid probability range (should already be valid for our params)
            prob1 = max(0.0, min(1.0, prob1))
            prob2 = max(0.0, min(1.0, prob2))
            
            # Symbol name: e.g., "B1_ell2" for ℓ=2, "B1_ell-1" for ℓ=-1
            if ell >= 0:
                symbol_name = f"{pair_name}_ell{ell}"
            else:
                symbol_name = f"{pair_name}_ell_neg{abs(ell)}"
            
            # Add to composite map
            composite_map[symbol_name] = [(base1, prob1), (base2, prob2)]
            
            # Add to symbol_to_idx
            symbol_to_idx[symbol_name] = current_idx
            
            # Create ideal vector [A_prob, C_prob, G_prob, T_prob]
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = prob1
            vec[idx2] = prob2
            ideal_vectors.append(vec)
            
            current_idx += 1
    
    return composite_map, symbol_to_idx, ideal_vectors


def print_alphabet_info(composite_map, symbol_to_idx, ideal_vectors):
    """Print detailed information about the alphabet."""
    print(f"\n🧬 Eta-Based Composite Alphabet (η={CONFIG['eta']}, ℓ∈{CONFIG['ell_values']}):")
    print(f"   Total Symbols: {len(symbol_to_idx)}")
    print(f"   Theoretical Capacity: {np.log2(len(symbol_to_idx)):.4f} bits/position")
    print(f"\n   {'Symbol':<16} {'Index':<6} {'Composition':<20} {'Ideal Vector [A,C,G,T]'}")
    print(f"   {'-'*70}")
    
    for sym, idx in sorted(symbol_to_idx.items(), key=lambda x: x[1]):
        comp_list = composite_map[sym]
        if len(comp_list) == 1:
            comp_str = comp_list[0][0]
        else:
            comp_str = f"{comp_list[0][0]}({comp_list[0][1]:.1f})|{comp_list[1][0]}({comp_list[1][1]:.1f})"
        vec = ideal_vectors[idx]
        vec_str = f"[{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}, {vec[3]:.2f}]"
        print(f"   {sym:<16} {idx:<6} {comp_str:<20} {vec_str}")


# Build the alphabet
COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS = build_eta_based_alphabet(CONFIG['eta'], CONFIG['ell_values'])
ALL_SYMBOLS = list(COMPOSITE_MAP.keys())

# Print first few and last few symbols for verification
print_alphabet_info(COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)



# =============================================================================
# CELL 5: ERROR RATES CLASS - EXTENDED VERSION
# =============================================================================

class ErrorRates:
    """Error rate configuration for multiple DNA sequencing technologies."""
    
    def __init__(self):
        self.general_errors = {'d': 0.0, 'ld': 0.0, 'i': 0.0, 's': 0.0}
        self.per_base_errors = {
            'A': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'C': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'G': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'T': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0}
        }
    
    def set_EZ17_values(self):
        """Erlich & Zielinski 2017 (Illumina MiSeq) error profile."""
        print("   >> Loading Erlich (EZ17) Error Profile...")
        # General errors
        self.general_errors = {
            's': 1.32e-03,
            'i': 5.81e-04,
            'd': 9.58e-04,
            'ld': 2.33e-04
        }
        # Per-base errors
        self.per_base_errors['A'] = {'s': 0.00135, 'i': 0.00057, 'd': 0.00099, 'ld': 0.00024}
        self.per_base_errors['C'] = {'s': 0.00135, 'i': 0.00059, 'd': 0.00098, 'ld': 0.00023}
        self.per_base_errors['G'] = {'s': 0.00126, 'i': 0.00059, 'd': 0.00094, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00132, 'i': 0.00058, 'd': 0.00096, 'ld': 0.00023}
    
    def set_G15_values(self):
        """Grass et al. 2015 (Illumina MiSeq + CustomArray) error profile."""
        print("   >> Loading Grass (G15) Error Profile...")
        # General errors
        self.general_errors = {
            's': 5.84e-03,
            'i': 8.57e-04,
            'd': 5.37e-03,
            'ld': 3.48e-04
        }
        # Per-base errors
        self.per_base_errors['A'] = {'s': 0.00605, 'i': 0.0009, 'd': 0.00543, 'ld': 0.00036}
        self.per_base_errors['C'] = {'s': 0.00563, 'i': 0.00083, 'd': 0.00513, 'ld': 0.00034}
        self.per_base_errors['G'] = {'s': 0.00577, 'i': 0.00085, 'd': 0.00539, 'ld': 0.00034}
        self.per_base_errors['T'] = {'s': 0.00591, 'i': 0.00084, 'd': 0.00559, 'ld': 0.00036}
    
    def set_O17_values(self):
        """Organick et al. 2017 (Illumina NextSeq + Twist) error profile."""
        print("   >> Loading Organick (O17) Error Profile...")
        # General errors
        self.general_errors = {
            's': 2.52e-03,
            'i': 4.14e-04,
            'd': 6.94e-04,
            'ld': 2.11e-04
        }
        # Per-base errors (converted from percentage format in original)
        self.per_base_errors['A'] = {'s': 0.00717, 'i': 0.0003, 'd': 0.00201, 'ld': 0.00054}
        self.per_base_errors['C'] = {'s': 0.00034, 'i': 0.00007, 'd': 0.00006, 'ld': 0.00001}
        self.per_base_errors['G'] = {'s': 0.00196, 'i': 0.00125, 'd': 0.00058, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00055, 'i': 0.00006, 'd': 0.00014, 'ld': 0.00006}
    
    def set_values_by_model(self, model_name):
        """Set error values based on model name."""
        if model_name == "erlich":
            self.set_EZ17_values()
        elif model_name == "grass":
            self.set_G15_values()
        elif model_name == "organick":
            self.set_O17_values()
        else:
            raise ValueError(f"Unknown error model: {model_name}")
    
    def print_current_values(self):
        print("\n   --- Error Configuration ---")
        print(f"   General: {self.general_errors}")
        for base in ['A', 'C', 'G', 'T']:
            rates = self.per_base_errors[base]
            print(f"   {base}: sub={rates['s']:.5f}, ins={rates['i']:.5f}, del={rates['d']:.5f}")
        print("   " + "-"*30)


# =============================================================================
# CELL 6: SEQUENCE GENERATION FUNCTIONS - UPDATED FOR ETA-BASED
# =============================================================================

def generate_composite_sequence(length, composite_map):
    """Generates a random sequence of composite symbols."""
    symbols = list(composite_map.keys())
    return [random.choice(symbols) for _ in range(length)]


def realize_sequence_eta(composite_seq, composite_map):
    """
    Converts composite symbols to a single DNA realization.
    
    For eta-based alphabet, each symbol has specific probabilities for its nucleotides.
    """
    realized = []
    for sym in composite_seq:
        components = composite_map[sym]
        
        if len(components) == 1:
            # Pure base
            nucleotide = components[0][0]
        else:
            # Mixed symbol - sample according to probabilities
            bases = [comp[0] for comp in components]
            probs = [comp[1] for comp in components]
            nucleotide = random.choices(bases, weights=probs, k=1)[0]
        
        realized.append(nucleotide)
    
    return "".join(realized)


def apply_ids_noise(sequence, error_profile):
    """
    Apply Insertion, Deletion, Substitution noise to a DNA sequence.
    (Same as previous version)
    """
    bases = ['A', 'C', 'G', 'T']
    noisy_seq = []
    
    for base in sequence:
        if base not in bases:
            continue
        
        rates = error_profile.per_base_errors[base]
        p_sub = rates['s']
        p_ins = rates['i']
        p_del = rates['d']
        
        # 1. DELETION Check
        if random.random() < p_del:
            continue
            
        # 2. INSERTION Check (pre-insertion)
        if random.random() < p_ins:
            noisy_seq.append(random.choice(bases))
            
        # 3. SUBSTITUTION vs MATCH Check
        if random.random() < p_sub:
            options = [b for b in bases if b != base]
            noisy_seq.append(random.choice(options))
        else:
            noisy_seq.append(base)
            
    return "".join(noisy_seq)


# =============================================================================
# CELL 7: MAIN DATASET GENERATION FUNCTION - UPDATED
# =============================================================================

def generate_dataset_eta(config, composite_map, symbol_to_idx, ideal_vectors):
    """
    Generate dataset with eta-based composite alphabet.
    """
    # Setup error profile
    errors = ErrorRates()
    errors.set_values_by_model(config['error_model'])
    errors.print_current_values()
    
    num_samples = config['num_samples']
    seq_length = config['seq_length']
    coverage = config['max_coverage']
    filename = config['dataset_path']
    
    # Initialize dataset structure
    dataset = {
        'metadata': {
            'type': f'Composite DNA Eta-Based (η={config["eta"]})',
            'error_profile': f'{config["error_specs"]["name"]}',
            'num_samples': num_samples,
            'seq_length': seq_length,
            'coverage_depth': coverage,
            'vocab_size': config['vocab_size'],
            'eta': config['eta'],
            'ell_values': config['ell_values'],
            'alphabet_mode': config['alphabet_mode'],
            'symbols': list(symbol_to_idx.keys()),
            'symbol_to_idx': symbol_to_idx,
            'ideal_vectors': ideal_vectors,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'seed': config['seed']
        },
        'data': []
    }
    
    print(f"\n{'='*60}")
    print(f"🔄 GENERATING DATASET")
    print(f"{'='*60}")
    print(f"   Samples: {num_samples:,}")
    print(f"   Sequence Length: {seq_length}")
    print(f"   Coverage Depth: {coverage}")
    print(f"   Alphabet: η={config['eta']}, {config['vocab_size']} classes")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    for i in range(num_samples):
        # A. Generate Ground Truth (Label) - composite sequence
        clean_composite_seq = generate_composite_sequence(seq_length, composite_map)
        
        # B. Generate Cluster (Input) - multiple noisy reads
        cluster_reads = []
        for _ in range(coverage):
            # 1. Realize: Composite -> DNA (using eta-based probabilities)
            realized_dna = realize_sequence_eta(clean_composite_seq, composite_map)
            # 2. Corrupt: DNA -> Noisy DNA
            noisy_read = apply_ids_noise(realized_dna, errors)
            cluster_reads.append(noisy_read)
            
        # C. Store sample
        sample = {
            'id': i,
            'label': clean_composite_seq,
            'cluster': cluster_reads
        }
        dataset['data'].append(sample)
        
        # Progress logging
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - start_time
            samples_per_sec = (i + 1) / elapsed
            eta_time = (num_samples - i - 1) / samples_per_sec
            print(f"   Processed {i+1:,}/{num_samples:,} | "
                  f"Speed: {samples_per_sec:.1f} samples/s | "
                  f"ETA: {eta_time:.1f}s")

    # Save dataset
    with open(filename, 'wb') as f:
        pickle.dump(dataset, f)
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*60}")
    print(f"✅ DATASET GENERATION COMPLETE")
    print(f"{'='*60}")
    print(f"   Output File: {filename}")
    print(f"   Total Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"   File Size: {os.path.getsize(filename) / (1024*1024):.1f} MB")
    
    # Print example
    print(f"\n🔍 Sample 0:")
    print(f"   Label (first 10): {dataset['data'][0]['label'][:10]}")
    print(f"   Read 1 (first 30): {dataset['data'][0]['cluster'][0][:30]}...")
    
    return dataset


# =============================================================================
# CELL 8: ANALYZE DATASET STATISTICS - UPDATED
# =============================================================================

def analyze_dataset_eta(dataset, symbol_to_idx, config):
    """Analyze and print dataset statistics for eta-based alphabet."""
    
    print(f"\n{'='*60}")
    print(f"📊 DATASET STATISTICS")
    print(f"{'='*60}")
    
    # Symbol distribution in labels
    all_symbols = []
    for sample in dataset['data']:
        all_symbols.extend(sample['label'])
    
    counter = Counter(all_symbols)
    total_symbols = len(all_symbols)
    
    print(f"\n🧬 Symbol Distribution in Labels:")
    print(f"   {'Symbol':<16} {'Count':>10} {'Percentage':>10}")
    print(f"   {'-'*40}")
    
    num_classes = len(symbol_to_idx)
    expected_pct = 100.0 / num_classes
    
    # Group by type
    pure_bases = ['A', 'C', 'G', 'T']
    pure_count = 0
    mix_count = 0
    
    for sym in sorted(counter.keys(), key=lambda x: symbol_to_idx[x]):
        count = counter[sym]
        pct = 100 * count / total_symbols
        
        if sym in pure_bases:
            pure_count += count
        else:
            mix_count += count
    
    print(f"\n📈 Symbol Type Distribution:")
    print(f"   Pure Bases (A,C,G,T): {pure_count:,} ({100*pure_count/total_symbols:.1f}%)")
    print(f"   Eta-Mix Symbols:      {mix_count:,} ({100*mix_count/total_symbols:.1f}%)")
    
    # Read length statistics
    all_read_lengths = []
    for sample in dataset['data']:
        for read in sample['cluster']:
            all_read_lengths.append(len(read))
    
    print(f"\n📏 Read Length Statistics:")
    print(f"   Original Length: {dataset['metadata']['seq_length']}")
    print(f"   Mean Read Length: {np.mean(all_read_lengths):.2f}")
    print(f"   Std Read Length: {np.std(all_read_lengths):.2f}")
    print(f"   Min Read Length: {np.min(all_read_lengths)}")
    print(f"   Max Read Length: {np.max(all_read_lengths)}")


# =============================================================================
# CELL 9: MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    
    print("\n" + "="*60)
    print(f"🧬 COMPOSITE DNA DATASET GENERATOR - ETA-BASED")
    print(f"   Error Model: {CONFIG['error_specs']['name']}")
    print(f"   Eta: {CONFIG['eta']}, Ell Values: {CONFIG['ell_values']}")
    print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
    print("="*60)
    
    # Check if dataset already exists
    if os.path.exists(CONFIG['dataset_path']):
        print(f"\n⚠️  Dataset already exists: {CONFIG['dataset_path']}")
        response = input("   Overwrite? (y/n): ").strip().lower()
        if response != 'y':
            print("   Aborted.")
            exit()
    
    # Generate dataset
    dataset = generate_dataset_eta(CONFIG, COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)
    
    # Analyze dataset
    analyze_dataset_eta(dataset, SYMBOL_TO_IDX, CONFIG)
    
    print(f"\n{'='*60}")
    print(f"🎉 Dataset generation completed successfully!")
    print(f"   Eta: {CONFIG['eta']}, Classes: {CONFIG['vocab_size']}")
    print(f"   File: {CONFIG['dataset_path']}")
    print(f"{'='*60}")

📋 DATASET GENERATION CONFIGURATION
   Error Model: ORGANICK (O17)
   Sequence Length: 77
   Eta: 0.2, Ell Values: [-2, -1, 0, 1, 2]
   Vocab Size: 34 classes
   Theoretical Capacity: 5.0875 bits/position
   Num Samples: 100,000
   Max Coverage: 50
   Output Path: ./dataset/dna_O17_eta0.2_100000_50.pkl
🎲 Random seed set to: 42

🧬 Eta-Based Composite Alphabet (η=0.2, ℓ∈[-2, -1, 0, 1, 2]):
   Total Symbols: 34
   Theoretical Capacity: 5.0875 bits/position

   Symbol           Index  Composition          Ideal Vector [A,C,G,T]
   ----------------------------------------------------------------------
   A                0      A                    [1.00, 0.00, 0.00, 0.00]
   C                1      C                    [0.00, 1.00, 0.00, 0.00]
   G                2      G                    [0.00, 0.00, 1.00, 0.00]
   T                3      T                    [0.00, 0.00, 0.00, 1.00]
   B1_ell_neg2      4      A(0.1)|C(0.9)        [0.10, 0.90, 0.00, 0.00]
   B1_ell_neg1      5      A(0.3